# E0: reproduce FLOCK zero-shot results

1. In the notebook settings, select the **GPU T4** accelerator and turn **Internet on**.
2. Choose **Save Version**, then **Save & Run All**.
3. Download the `e0_*` files from the Output tab.

The notebook checks the walker, runs three small inductive graphs for comparison with Table 9, and
times 20 Hetionet triples at the paper's zero-shot setting (n = 512, P = 2).

In [ ]:
import base64, io, json, os, subprocess, tarfile
from pathlib import Path

EXPERIMENT_COMMIT = "62a801277c7523aca45366473d0621a199c21de7"
UPSTREAM_COMMIT = "f35103d25a78bdf4075de5c673a51de4979aa4d7"
WORK = Path(os.environ.get("E0_WORK", "/tmp/e0"))
OUTPUT = Path(os.environ.get("E0_OUTPUT", "/kaggle/working"))
REPO = WORK / "flock"
PYTHON = Path(os.environ.get("E0_PYTHON", WORK / "env/bin/python"))
SCRIPTS = REPO / "experiments/e0_reproduction"
RUN_ENV = {
    **os.environ,
    "EXPERIMENT_COMMIT": EXPERIMENT_COMMIT,
    "CUDA_VISIBLE_DEVICES": "0",
    "TQDM_DISABLE": "1",
}
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)


def sh(command, check=True, cwd=None):
    print("$", command, flush=True)
    code = subprocess.run(f"set -o pipefail; {command}", shell=True, executable="/bin/bash", cwd=cwd, env=RUN_ENV).returncode
    if code and check:
        raise RuntimeError(f"command failed with exit code {code}")
    if code:
        print(f"FAILED with exit code {code}; continuing", flush=True)
    return code == 0

In [ ]:
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
).stdout
print(gpu)
assert "T4" in gpu, "Select the GPU T4 accelerator in the notebook settings."
sh("nproc")

In [ ]:
if not REPO.exists():
    sh(f"git clone -q https://github.com/jw9730/flock {REPO}")
sh(f"git -C {REPO} checkout -q {UPSTREAM_COMMIT}")
BUNDLE = "H4sIAAAAAAAAA+19+3PbRrZmfuZfgeKt3aVmKAbvh2u4tR5bTrxjOy7Hmbm3dLUIREIiIpLgEKBtzdzkb7/ndDeArwFSAhU7M1sjVmIRQHeffpw+r+80uEk+xdfL/DJZxos0mafbrz7/x6SP7/viL33af+mh+5XlOaZlBqYbuHTfcj3L+er6C/Sl89kVZbIlkts8L+8qd9/z9uD+P/l4tjHLV6t0XU59OwlNyw6CWeDZTjJLXM/xfTdw5qZvW4kVRTPbmqfB4B/d58fP5/uknzbpNuP1L77+UjR4PwSBd3j/0/fW/ndc2/7K+1Idws+/+P7H9U/NeJtutvl8NyuzfP3Z+OEB6++TBHpc/9/gc9f6zxbp7Cb+mCxv0u1kc/tgGlL/uwfW3/ICWmt9/W3Ls4OvfpNJ/Bdf/+Fw+CxfbZJtapSL1NimyzQp0rnBq05Xs3w7z9bXxsesXBhPN5t0Pc8+GU8n9thI1nNDcAhVTEqjSNO5qldQxU2alJPB4N1uXaiGk6Xx44tlPruZiDI/GvlaPLneJpuFkV+Ji01C7Pi/CuNFdr2jLllEphQP/pZu89NikZfGtykzZ8oUy5L6Nhm8p+fzpFgQ+WW2JvLZWvbpSrbCg/vrjtg8LcZGkXN7t+LmmpqjoZcVddGVifHUePb2B+Nyly3n6skgXX/Itvma94lsnUZEDGG8O3v6/PWZkRVGus531wsa8g9Fcp0+MTa35YJG2NpDxvnpKc+UYZm2e8FX+a7c7EpjMvl6mxa7pdyGsnwsKk9+KvL1xYAWajDIVpucuptsr6nbRVpdL2jwy+yyuuQK1fe8qL5tlkl5lW9X1XWxu6S9PkuLukRxWwyutvmK5qTk9gx1/y1d1rR/ytY/JXZ1td6taFBJYaw31a0y384W2kV8neartNxms+r2bbJaSlKtApN5UiYV3ef0nVjo7O13xlR0YhTHV9kyjeOTCU1WvvyQjk4mzLskvc7ti8Gz7968ePkNFR4W21lMd7Py9utZviY++JoZiPknLrfJuhAi7kP69dqzbCpYpKvLZWpPuF9Dovjq6fuX3735nlo6Hy6zm3Q4NobzrKi+XhErred/oT0xvBi8+e75mSz5aicff5usv8+XOX89WxE/51v++h/5POG/z4nhFn9mV4uvni3Sj5fJbCYe/d/k8lJ8eZVmCTV99vwb2fTAoM+oJlD1oyJ0Mm4VgL5WPeiUgUFUvburHeh2p1hVphlMXQTmAhqTA60LwSwdLqTPW1VIzFRdRpvNdpGLwWAwT6+UVIitWGz30ckTUZn21x/Fjm9kUiJFF7MkyRpjtkyKIi2Mef7ESJPZwrhKZrytdoWRGFfLjKQjScSc9oOQluXthoSE8Xvjv97914R3L1NhH7vgFRVcM8nW8/TTiG+eGLQ3xeOxEdN/LGXE8l+IemWSLTv1+KasF4tKfN2uR53gemKbTUri9Hw7Oq8ZXDVEUj9hoVo3Vt3QOnIiWkzn12ksqnWaFYOT3ShU6W1a7rZrsZNH4obexFQ2MEvK0XlzdwwlJjyxI/PkYmzMs9XUUktdN8MDxFbEgMdq3L83lul6VI/25AJqk+SK1/k8LaZcRkxr62k1CcVUG6Zt/K7VrKp3ovhrmSfzeEVNLyveIsE6YZlK012k23JkkhoqtyMh2b5GYTU8kdMmROOVUJaioaISiUKBDuTSpisW6Sktg5TJk/fqzqhqWcpDFpbUozL9VI5U+1ImUk0WeJMiuUpj7vWoapNq0ORvR4r1p0Opjy3aTtebXTE9N2k90k0+WxRTGswlrQH9md1sqOialHM1DrX8otcjSfR8KEY0vDiZpB8SmiI1a0VCpNOReDgWe26stgPry+kbalV+jYXOmb5IlkWq5je7Ek9YDbNK57JP6qWUi7dK1rtkGXOxEf8j+7feTEgbzPPVpH5QNSbJGDT3qWhQVpA7W7S4zlmAzKs1huGKMUgrZyQH0t0nFyfVwOfZdVqUI2lrqcZYoadbWh+l2SfFIrE9f6S4gzapbIs3p6zYdELWnew2c2YFWW4iBkPKsswvb8u0GOnro6os0k+qM1XfFvkmJvu4zIoymxWqj2NhTN3W+2MqJW0jR1+TCcl21yL/yC0UwtaSddP5xHibFxnXMyxesOJGCs7LdJbsaK55V5+yTV7W1uQVmZxZWctQcXOsZF6s9vrYWCcrWjV1Mc+Inti8LDTVJFxRf+krbz/7pJqSvbMphee6EQHUir7n1fTJx/SUOOlqt1yORiBNjOY7XZxaslJynZAYKGUdYZXcUWku5Ntlni+bpZe7gofbEvnAA7zhs7l8Tl+IWEfhjI2OLgFGlgM7bzd0QS3VD9W9cUWOH7ZVC3ezaVeNfW/N99tdKiXbZpt+yPIdLeNst2Xjjp6KRT9/MjasJ6cWCZ/62n4iF4tZFdaj7mSnsaZ4sxJVvw4UrniXNR7zF9LNClJUH1KyxgVVVfB/T5F7qoJkmtx8TLbcTsOgqjFjOjWsSpqNtNH8wTBPJsmamZWWnyXcqFv9D8zTyXKpyaMko26RD1aSh3+2JRNrNLzKd+S4JTwB0klSQjMxCnKmlnIDVg5R5YZJg2h4ItcnWeZrVh6/wCRKPUeLjYuAo+B+8tLxONv3LTWXor7gVtjMcoCqZuu+oypKgSRkzbTFCV2W1KWXJgj/Xk/dkBsbPqG9VWqLMSmyv6VgKAyrJa9LxLTQNHfsZwi2mMesEJ8YvIlHo5pD/ocuXvRuq5VEOmKnlHksto7qXNPdVldikhPMaFSGrAiyi0a/NHx6Ltbvwvifxi/Ak+ruyWSVJmuNdLtt1UxdsyGyj0YvEj/DQMUAaaBiwPcOtOpMZ8DQl1/2D/iX3iPujvSXvc33ax0HS3qdXJFVUs4WtPviREVY4sRuaI1QxExRcp0QzZEmV/THe4gPSxKzcS0+Yh4lNZ2Q+xuz1Sn2Vvwp3zZlat7Fjvw/HOxJm3CXfcn4K9LZjp3uWBMTMWw07cGk2K36tiGnkY0z+SAhI/R2RYKcpEk9kUq6nGtVL/ZMkS4gRAdjsh2ur1PggIo9zhvZc9FD2rSW5OfK+mWLU4R7ikMmsJLrdYSO7Bplrd1jOovKyrUTJk6yvk5H9omSuTJud1RrmiHOevtw813BKiM+ooF8veSp4Vghs0A1NPIueCbrS9L37epktEvysqGmDTkc1YK60OrLeyLCVsTSrMZq+rKs0jLhGRjBAvD966wc/S7ZXu9E4LzrATSxtcl2tx6dD6kC+U5NFbJgZh/nU/bT6FuyKTkiIUOBYkLJ4CJ3TX0VfCHneVKUcypGf7bkFit9/G8GR0D/RAy6FDHNlLbrjbFbb2h7cgiDNPyMVtPgjU56fkbTujZ2G2ojTVZ0SfpJRHM3Mr6RlWSV57vtLBXQMFne0mfkXtDMsSMrrGr0W8fGUJgIpzJyObxQbiZXp7J5MVER1Ml1Wo6GZ//+9uzdy9dnb97Hz757/frl++FJZfqoOtRn3Y2r2+K5J+H84VSEQEUY7Ozp82FjZGodnUiROhreAXQMD9kAkiaxh/yiiaJ5yj5+dpURJbKcZmlclxbSUnSTfaddwX08PSX/fUZSIFvLS2YHraeaCGJ+U2ypC0DyoOm+dOzxyWZHRgf7+UWcfCANygKdCrIzQZNfEKV5TFOfXF1la1qwkamRk8FqKl8FiSfyTszCnmZodNLegVRWOrVxVSaOO7u0DutWpSHQu7+e2NNUmlyjToF6U9IcVlau4AF2lKuQ+OSp2mJvxZPRPC1mtFWEnxrH83xGrbGsWrEruI1FTG9aV36XfHzeVPg2XW5eVEVPgN4kmc/jai+PhjKkP5Se6JQU2ZiFRLJbllMO899ZU+55xYH0gHeWKir+cOGi2uhCJrMrChEmKZM4aj7tRDbrkGPL/5OB24rrWY9RgXvkPvdjsl/4q3YEekENwQaq5CetaC1K64aQp1hsxNILZ6G8Ebyb1WEREswTcbtreZPZRGX3BiraG0oWvUfVtrv3s3JJ2T5hYGUyp4EzBTHcscEzui6n9kktwUQLcl0b8SXgC3hyMvm4zcpURuXubNf4vTH8T5ZRgwE1HsfsAsUxq7dhHPNeiOOhpCM3xnHpOXfhv9VFGnOYImb05EEw8D34r+u7fhv/dSz+ezyp4z+P+O+73VrHfqVOl3p0kxPjA/oqUF8FGJM6FvHQ96xsjEjDcAUQmkoApTY1VDh5rAQZt1WSySmtk916tmDTkfagjLJil1TAmo1MAb1IpHbw4+npKvl0ysbQMi1+NAra9inDMTKqy3aYoPdxkVElsil2RXq1W8qYX7ZiZPtyV4r4hxyUGAkOqgZ0pRiQoO7+jWGcnqoBFsZf3ljhu3cv1wpnfPLB4qfph4yNqs2OLi7Z5zvlaIJhGf9Zy4kDgHBDR4C7DwSDl/n1NY25upSTVF+l0uhDNLhCcUlA3I8MH8SCRU0EwqvqlU74ElDwt2fP/vT2u5dv3ougpARaGo4uvhboijJfJ5tyMRy8fPP8h2fvX/75LO6FI2frGkS2/BpDtnwFIr9/9/TN90c1eD8w/f7pH1+dxVH85ulrgQtLTTt88UfLu7GdANmNFB3dP6UvUo0NuxzJRf7yBoq8OXv1ql3gzSsoUKVe8IP6+3jw8+D9d6/OaLzPzqhT5sS0lKVWbfzPgINVjEDe/7IPRFYXIf+resKyZoCmPjc2lhXG6mnjjadkftJ2p/2C3l8bLjqII8kHd4FPCrXczRMsEHPoRBYCOFHyykh2WIb+eXvErJXY2mTBMjaETIlFhPJJbRcKAUhGr7KmVXCV90opYJjR8MmwMlFJ7k2NfZxLlkcFg7PhUS++xMfaO6fCGNm40Q1C1QgxUNW15hkji8xZZOwNBUEpLpkeT5KiRS7CcjmEahKGpIom3LwUFmSnLCOUbAIS0zUCQjP0iK6aKHQ9xUjOh+rJkOEK9V0O9YrD4bw4E1wtxdc8reOqkan6e1JVnKiJmIh8oqnoW720TSkSDtl6wjwaN6tMxZsLZGyqMRZ065SHJbkwHGGUnoF4XjOHaJWvDqdCkGq/Wecf1yLnoTDST/RnKYT9j7TByAxdkxC9/dGY56R+hXZdpEWdOlHU2B07+DyapLg5HzIfDkWgZliLnZfrq5SE+CwddoIqehIBf/YnEtTDmWBKQXOX7D3yhOPm4Z7cgrr1Tn6B3jg/PtQ2P7totdlkHTR16ntN0X7JExXnHBjWvkyJw1XESAbdfqoa7IG1Oloj92RkSd+xHhNZeMmnWJlliqvW1TWjaIcXQ/p559ZFxS/QUhUVYhgMbwu0TV102KamJRvUMldYaJOxuhrV1U/On0DLKkQqrEit2yJophxs+bg7krvHyfCV5L5DjYgEokNt8MNzqN8EHqkZtTDCEI8/JMtdWkggVk5OyaYtL4PwO1liFXXKiBYnkw0k62R5W2TVpawt7NDhntSSbf6RVQzf4a8sC/gvJ02KiufDiPY8rSvdPR/OCykANMvmnPuqJxD9fbjabjlEzJVeDGnHDhdZWVhmfW8xvKgCRJxXsuPkB6ko2esWLseuYUapSKUNqIIAwhOZVuVEKbaXRXRJGc4cvHwl7o2Gqam0ZttYaAIIlfAeK/clVgr2kEYXNUGty2ul28UFKHgVi5YqUuoeZg55hwV8ExmSzCuKiKTWWN1UakDUmMo/yjKpWI6BarX/7YtJmY+wlFAIsVQI094qpt1KtVgiBLVPkoiBozjRY2GwhjKcExPlKSsZcXXSFJa6maNEDIDNypGUASLxqbEGWHRtYmpU5baQtzYkBzLNrhdlIRALlXrUJDJp/ZGkWoOsokJtk6adpSSMQfJnaCE3aULDSlf59lb0uQ4CitwYlgsZW3Hp9opYaUc2xXbUJ0VJ+laVDJarpDZAvRKNQSk3wFgtbio5ZwoLP1YbNFbtQkCwSInFRbLjnp4ap3Icg45qqHtxn2rgdJuYrfsp1j9Ct6xkmhIzXiVeJLanxnIubpIGB1nTKsD3/w89uDhRYUKWkCKC25a8BxCGxh4WImBPqB9ERyteWsQcsa85XSSccVgWbWPlQMYbraSYIrnTCjSaa+lCpVvyBmP7XFtNOM1r3Iyhz9Qfaqhed56L6i4UrpabF0F9RQBDaiq+X+VUMJrBd6FUteIiHC2/4tMtg95Xtf1Zl1GMQDwrGlSXuBoZi/64zJcpWRSibnJZjO6uf2L8gfRe7TiTiKg5uk461ALYYjuJALb41n0WC/UsZq4pRjp933QK8aIwykrMXF8KEFaGgVibbHcc+hrVt9798P3Tb87i789evSDNvyOz/9O2YAoMc/zud3a7ffLotLY1j/dT9YhWjaUt+cYndVPOQZHZnpx/YlSo8n6GDR60N1Yz7sZmxp1YzN2UxCQNGSrO6UsxPVd6S87ZBXRA3LizsSYyuR/SurMyhGOh9hcCz3iCT9lWqqq/Fa52VX+fSctVvr65vnsGFCgHjfbG53ZSizWRL+V1XOVsG3QxsPrxZJNvRpXYb26fQ9CySle46CYGN/aLtMirLF8tp7dpU/HLRe0IiT3Z2I8CZBqZJ/fsQsFkbfQPQT+mRkxIMo0F1/mF1JLsEzABdgpqm1eECYrNkmHz8VBLqRBNn8tWLipA/04jv0kGOATcVc2di2zSGmxjn2BXLMCMEZN2ANSreEI9+CeB9I763OHufTYaDzn/6zru4/nf3+BzjLv/UBp347+81J3zvzb9/4j//gafvw+GEYvmAX0R1iWjRrY3HA+GQhjzndoOMUjQbvnRDxxunzhhyN8X4sJ36ft7eT9y+Lu679H3F+Kra3KZF+q+7w5+HmtkPbMfWQfIeq7T0HXchq7nBjVhx7Mbwp7vtwkH/cbrmg6M12wG7JoWDNiymhFbIYzYttuELbPfkF03wpm2G8qOD5QdmGscsu87GuW//Kn3Els+zLXTEHZMryHsRn5N2IYRu5HVpttzjS0f6Dp2M9XygaLrWE5N1xIzoQjbQdgm3HONHcGv9YCDZsB+CMwFhJ0QmcvpEO67xpbYQYqyHfrNkEOgbEdRM+QwaCg7pr6f3rw67bmdBDtVI7ZhO4WwxJ4bwYgt4C3Tb9Pty1sRTrUfNYSDAAiHQNj1YKqjNt2evOWaAdBtVliKp3q8YUPWB9byfK9NtzdraXILWMsD8eHZzSZ2LJhoz+4scG/xEVggPiQ1+SBE8RE0u1gXH5EuuF6/t4wy+dSLtG0DVyuhoQYNXK3IKdYH0k7odEgv0mTJh9b7UI9gyp0AVFTgI3VQFSGwmBtaLeq2kW+ve5E2kbstr+FuM4LVtrxmzmGtLb89bNsoZlm/MXs4Zg/GbINadn1kM5TZXtgi7ZCPU/YkDWrKNRttYYewtVy3UZCOCaN2W5qZSZMHuU16EfctIB54Daf5HsixIGrW2g/BHgnt9lq7/ac8ACaXulqNO8IZPzTq9lq7x7C4jxssAIXloxQl9wuG3RBv6cnXnCGerPoN2tGsP+AzFxg8tMAmCUCORo7XsoY45NVPnIWwtXwPrKEADc9mvj0TRErQNsJEftNeqkASBYnvNcrZs4C5/LARYyBG/KhL076fpmeBYm7WVSqIal0bO8RzgCQxQoek02eYmqJo1tTTLFy/Uckeqig/bK/p6Qe3B9UQjD3Ftqo7uJ5hAAuKYzV19VTltN1D1XdhSQNp3anlgl0bRI2w8iPYOaHpdKj2WFQyxZEqSKkA9GEQIlXYNUHUIdpjWaW2q+RT0HCv9E9qkwccFx9MS98MO1R7LKt0hKqhguqXM18N1W5Y2PccGKrdMfD6LGuAwiGyGnM2NEHlh1EIVHFZW9KYqfbZq7aPY4Xd6iMz+Q1VL0Qx3JIQTLXHukrhU1NtBKGHxlXggpCIUPZ3h9pntwaoZy0g6iBRsC88D3drqCv5b18b1s39VE00LcyooWoGdkPVspsJNn0Yq2VFbapOH6oOuAomSCYTl9WyIqAaINWwTdXrRdVFqhZQxbFaQNRFop0JJirZdd5jaYGu2qTyvhZlcEGj27Bh/Zbv+/LV22fG96tEpGDeJ6FM0HUuxnRMNFo9MFpNWF635QwK0q8Yg76ftI02K0QaHBPsJxcYSzeXW4NmIyaKdIfwPSR2oxuKoRUYsosc7dsNb7kRxs88r0PYbvH0fsK2o1npzTJr1irYybYDq+xELWXwnN8bkW57EJaRkopw1BCWRkQ1YhNGrPm/OuFn+XqWbso3aUlucJ9xm0je8hsRYqGZ4QB7267mGDkt+s/PPu3h7wOrHQBx3w8OrHYz6S6OnczrPbS7DH5g3tFSdzHQ4mAQz8dACxjMbitS+x9Pv/nOscw+g/bQnLQaC8A1gdV8G0ZtIYsHOuGn26wo8z+7PShbuNYyolAF04DVZJStkie41q6+1s//uEnnWdKT0Rx0Fjw0t2yUo2ajljGuRdOhy+/moMW9O9sDdeUE4IcGuLUh1uKCbekEVst6t7ybU9sJRpZ50oc6GtQObG9JpdpfkbOfvGu2w+MVebsfedxhSmPJB6irwRayAwdXvA1HVNS9XtQddMOlNVhFXbTgHoQ+LFBgHlsKgwu6IdLMHgjHIDzhQKjtIB6DbNeFRXrjMTh0p1n3O/AYMJK8jqfaF44BqsBth8EYG4frtKn2x2I0Kwn0to7FwO72cZo7AEHf5bUtnGZYXtTawN7orXqtKP0RSIyGSjRy9CAOY8Pa2h2qfWPlKEHBGjwEwmis7Hkdsv1BmBCHezwI0wFDeu4htEI9iNAfBGF0aNHq+Mo9uUoHQyKQW4dAGF1q2J0B90VhLFziCG0DHDFa/BFag77dJtyXtTBY6/VBYVywSTz34SgMBuZl7LlCYdAEhICTDIxVjrOlUz4GhXFAM7iARTgRLrLZ+DiuhbqxFZ04EoTRrU+Y8IMYDPo5JNIejMFYJiy1Bd57BJtZegJqM5vovQftCe8PwjgY+3JxPzsal4H9qYGMttci3R+EcUwt2NcITztEOwhBGFzrDuB2FAaDBmhoIhgBLA7BzQC9+NBpQ0/9IRgHczPcAIwQLVEBAvTSYqv1Y9QhfQSLB8hoMgL5Xj2AccsHauAYN29FdY9BYaKonnFrYtZ0o2Zjy9sv1O3ar6PbD8df0K3zAU7GGLYPottzcLRWJ/OnT2DXMzEHBiKsBxEYF4Oddtu56RnXxYVtRnoQhNEi2F53oH2iushMEi+4D4TBpBs/6hjyvcK6EYIwqJ0OgjC4pg/EYDBYH0TNUAPctQEE6wMLrI/QsjpU+2AwWgQb3GQ/1HYrSCnEDyOvQ7QPtIYawXMxKKBZeGDTauG+VlSgNwaD/NuEQXycXrvZM76PoXrXats6vRAYTQmAKA4xwSYEGzrwYX6jVpyvLwITICuBUPIC2DWBjJSruQGqYddy77VXNTgPkg50ql4zVt/0kerDIBgfGdgBCYE2bBCALAy0serr2heCwUQxACUOAjABZrS0oqe9AZiGpAk5NP3gly7NXvCLDZxkAjBsBi5SRagJR9oh2ht+cTBG7DTDPYy/aE6go1M+Dn/RPIRe+AsmSrU03TH4iwy81nE7MJWBl+X9ynADwm0R1Rt+0c0JSOgwtbkGwBZtGL+dBdgffcEIgwtRHKTrOQcSaltkjwBfXAsdfRsHrMFNDW9hPonntsGPI8EXB7WfDR4ohghBOOsp223k5yjkBdcZIjou5qRpuY8+gpruPtq9kRf3kFeCKRAuOEQt5EVX+v2RFw91MNqtmJulzBpV4WCK1BGwiz5ejCa5Wro45hLDZHv2r4FdpJSs0/MhMOtBlMMHbeFikkuL9hGwixNh8A4D4BFytwPcjTBXe1Mfjbt4mH1oNYtteyi7IbpjI+zya1EXLfWxkWc25uJhtMHGrMs2cH887II55BCfdmyQah5YuI6F2IPVBh8UeY30220qXr/Ex1RrwloyNx5+wcC45yGmipzutP0lPgF8L1VE2KRpX/nAKMsA88DMG78VFpei7DXtsN3q/vEiuqa8XzVedF8Q30L/0GWnVOJb/Fq5B+Fb0vesp7zpQqg5i3azzcIInUWnzWp9Q9U+usbSz1BeKoY7QKhhalfYOfTTM07taygyuBYhsnYAsFpkAoeFHUCzd6AaccQA9lSEUGbYiJNIc9+6J2B6rm+o6axGd4QI2YeQLhh5mtsYten2XN7QPxCECBFtCsHelxNRTXRrhfuDXBGGBELgq8jCVMXQOTDTrRU+AuYKcSuFkMMXYdJTCCGmCI2iyOmkK/bcSejheCbGIzRgD0IDqCm9oAO+9JYdmrhuCAcuWr9WIzsCG6KHvtNBm/rKDo1uY/cGeCjBB3M/ROQl6MZCevJWgMdPfMiXDMJDQsvRQpedcEhf3grQBkTNGKLBHcCRwchBECBsR+L7w1yRlvsLIiQKYLYjD3YyguTRr8O5Ag2ghwhUhKZQBBhuhIhm1EFe+gNdEVphEcBNER7wiiBQ3SL9cKQrwtzyCEftuzhqC0h7OOfts2X9ka4QcYgAUkBCBM1DgHIjhHJJ1v0KqEuK4mrckI6BNn9kAu0wwhl/MNIVIZgbRjjhuNZuI8okd1SUvfaoj0K6Qi3R3WoEWuSg5ohwi93B40cgXRZCMQ3SZeNxOhPTfYCubbfTfXqDXZrN2QizQE/la5ZZF6N7juH0OpqCBqd9ABbxUHijmgw6o+0Hi2jncRuALbAcJNrQRPc58Lv4Wo/4uQbFeBB31PUyMLOWEhF2aPbBRLSEhLARGwFuIB8S5ULMFAyi7hmcHmsaYnQCTqIGkWaAgIRGfDr0ukdweixp6KEpD9YHymYPcAKdaPu1Af2QrkA/Xw2MhMEnD87CSNlRGwAPOm2EZ8hCCP60hnqIZhf96bNPcU3VgWb5ACNOMpxecS+G2ZwuvtZjUQPEYfD4o+6OgkkZoV/YPjfQD+jyEXLyXBSEaMhaDSuFiNTuG2qfrarlsoTAwKhlMWUqxNOtYUvDH4OISNVdMRSkeUYWYE8h7h0MtkStfINjEJFQgzKjZtQhjjqEkyERHlSIzM6poz7wno+a3YXXE6gUBPUAXCQfUx+9Ns0+8J4XaTnbgMTjrnXB9db9QbdzqqsPwOfj0rqwgbQsJTzyE2AKrxc9GOHz0WL1IBEv9DCmA6on8vDMtP1gtAtjOi6cEvdQaGASnnY43nvwYaOW3ABl66GyBcKhlivUSvM8AvDy9ddtNM6gfrwKRIetWW6/8rSRhUttho38wHgpnrkJIi1Wuwd26ot4wYzDyRflm6HDUjkmGFQKunT7ol0h7uYQjtJFCDGGDihhNKxaeTX9wS7NXHZR94ea5QoGB6YmRC1s8wi8S4NebBPTtkFJ2HgWNgKZ6bZStY7Du7Q3FUFyRGBpmdugoPDESdgy2I/Au3ztwAesc4jHuqQZWOknIHwI8emHdumRPIjFRwgCgB8YYY5EGLST446Du3QfCSy+EIHsACNMepC67ZkdB3cFGogPvn+IudQhWGER2kORW6Mw/PvCD0RhcPEhO/EgChNpOb7d4z5fDoXBKP1DYZgAt7gP4K4GwyDdCFN7w24OaO8gPZ6eDCB2fBiHQQPU6eAhfXGYCMEBB8U4htUgfqvxmNdBJX41DqMBQMBYeKCrvbM/BwyDIS3wMfR5dh/8xrcII1ohuOct/AdUdTDWN/NnBmEwuoRGt5b6ZHawkJ6MFeDbFHwbHFftOLDZcFaABwVJ7TwQhAkxxSyAFzcGmNHnA9DWiqd1nNcHoTCQ/4Pncf0AbFB05IIu3NVbdmivqgwOhMYDOC7Yiod0kZDeIAyYnxHkkUeooFFcagHiTmD8KAgmwmN0IRwSuAOCwdh42KZ+BASD9n4EejEKUYoEYX/S/SGYAEkDIgBcFqHg/FwITIShvhBwpxAHHdpoDgFl69e87y3CV4RGLq41jBpeLBjhi5SizrGyIxAYNAkiSGaM0LXUSGOWW+R3wZ9jeBzNgggRGB3riw4td+s9d0dAMFrSkwVpAzY6tSbafprGih78yjct+wiAXT1k74DxF6DRaXbBiT7hXbT1rT4ojPaio+6B+n6HU5Cq2wOGwSgcmf4Pg2HQr4DX2x2GYbT57U5vn+AuRoYOLSmiMNrrGs3ueZg+KAwe3g88tAQwbQ+NPUyWDPe8f60PDIMOc+DjWJGTAjRtcVH9B7317TAO42ljBaqacO6ereoFw2iYPCbpoanlwQRjXmQYdQ//9DlyhO/+9MDA80ADOpCDEKJAbu/UnjhMK8AIwVx8nREuquYYPuiNb3qoC6yrQMtxdhFx0t4z1wV/eiwqJua5IPb1TEQMKWqHnNw9L+fqi8LYGFLEjJ7DKAyqWmvPK8m+HAqjk+4gIl8GhQkQIfDaUb0vhsNgdlw7df7LwTCISUSdofaFYQKcZMyjDvGFWAG8sTdCAChqn9rrDcPgaUG0KA6iMAFmA3qtNK0jUBgtaArnQfQ0RAhbhr5mPz342FGg/WQC2FAYw8I3gIX4Au6wtYmORmEkIL0v18PC8wGedYj+nne+9YZhEKcHM13PiwMPHOOVkf8rgJiWW9aQNjGh2cW40ufAYaIDiWmhlisF72XWw1m65fo5UBgX3/Ph4Dt18WwCGdlfAoXBqDQCuogyti27I0AYPZgG8GaESjmEA30RHi0LO++k/tIwjJZ5eejE069FYfBNcwHGWvAdv5Fz6EV3PVEYWzuGAwck8LBsiCCQxujtrLW+Z460aGJgIiCAFgnCq6i1vLbH1OvIUWRpQWpQHQiu+nC4TceT/T1Ads8zRxFa9SEEeBwN/0B5oiW62gx2DX7+R//E0j/1B3//a5v+dZdtU3Fxer3ZTcpPd/7kVd/P3b//ZVqW57V//y3wTOvx979+g8+/Gd+8/cFI1x+ybb7mlRe/zlguyF5uOGMsflQ635XG29v3/HuRY/mzkafFTPwKqpGs56IO/wQx/170x0U2WxjrNJ0bz354/vS02KSz7CqbGeL3x4vJYJZuS7oxndqm7U8si9T2YLbgH9QsT9f846rL7G/pdjp1Jt7EGcyW+W6+yWY3y5RvWXQrTYpb/hHv6dSaWO7gKlum/EPg06k7sSd0XTBJ1Xw0MQfZfJ1wXaLzU7b+KbFlQ/7gp/xymV1yMz4VWyXbm92mSK4EITIaBqvNKikX/JyuB+tk+3GRLAtqeWJzu6SsP+bbm09cPBisd6vNLZclqu5gk8xuEv7deCrtU1Obgn+DfToNqIv2YHN7ma3nliU7YtI1/6CoKO0QLSp+e5usqDj1ixrjzZkWpaDs0CAHxSy7ycrTZZps10wymlh8T9K3Ar5Ky92mzHPubhAKIsWt6qDl0lW54B8L3VCJWblksiHfFCt7nebyV66Znij71/mKZzcwqeXydkMdPU0/lem64B9b5icWT+Buu6TpdLgWN/ZxkabUMtvedPWPZvbHz+Pn8fP4efw8fh4/j5/Hz+Pn8fP4efw8fh4/j5/Hz+PnX+zz38fpVQUAyAAA"
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as archive:
    archive.extractall(REPO)
print("code ready at", REPO)

In [ ]:
sh("pip install -q uv")
sh(f"uv venv -q --python 3.12 {WORK / 'env'}")
sh(f"uv pip install -q --python {PYTHON} --index-url https://download.pytorch.org/whl/cu126 torch==2.8.0")
sh(f"uv pip install -q --python {PYTHON} -r {REPO}/experiments/requirements-gpu.txt")
sh(
    f"uv pip install -q --python {PYTHON} --only-binary :all: torch-scatter==2.1.2 "
    "-f https://data.pyg.org/whl/torch-2.8.0+cu126.html"
)
sh(f"uv pip install -q --python {PYTHON} --no-deps .", cwd=REPO / "graph-walker")
sh(f"{PYTHON} -c 'import torch, graph_walker; print(torch.__version__, torch.cuda.get_device_name(0))'", cwd=WORK)
sh(f"uv pip freeze --python {PYTHON} > {OUTPUT}/e0_gpu_freeze.txt")

In [ ]:
sh(
    f"{PYTHON} check_walker.py --output {OUTPUT}/e0_walker_check_kaggle.json "
    f"2>&1 | tee {OUTPUT}/e0_walker_check_kaggle.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 8 --data-root {WORK}/data "
    "--datasets FB15k237Inductive:v1,WN18RRInductive:v1,NELLInductive:v1 "
    f"--output {OUTPUT}/e0_zero_shot.json 2>&1 | tee {OUTPUT}/e0_zero_shot.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
sh(
    f"{PYTHON} reproduce_zero_shot.py --device cuda --batch-size 1 --max-triples 20 --data-root {WORK}/data "
    f"--datasets Hetionet --output {OUTPUT}/e0_hetionet_timing.json 2>&1 | tee {OUTPUT}/e0_hetionet_timing.log",
    check=False,
    cwd=SCRIPTS,
)

In [ ]:
for name in ("e0_zero_shot.json", "e0_hetionet_timing.json"):
    path = OUTPUT / name
    if not path.exists():
        print(name, "is missing; check its log")
        continue
    for run in json.loads(path.read_text())["runs"]:
        print(
            run["dataset"], "| MRR", round(run["measured"]["mrr"], 3), "| paper", run["paper_all_relations"]["mrr"],
            "| full run", run["full_run"], "| s/triple", round(run["seconds_per_triple"], 2),
            "| peak GPU GB", round(run["peak_gpu_memory_gb"], 2),
        )